# Data Acquisition: Shape of the Raw Data

Goal: prove we pulled each dataset down and that it opens in a dataframe, and note the shape of what we got.

For each dataset:
1. **Load** it into a polars DataFrame
2. **Row / column counts**
3. **What a row represents**
4. **Key columns + types**
5. **Time range**
6. **Geography**

Raw files live in `data/raw/` (gitignored). To get them, run from the project root:

```
uv run python scripts/fetch_alpr_cameras.py --state Illinois --out data/raw/Illinois_alpr.geojson
uv run python scripts/fetch_eyesonflock.py
uv run python scripts/fetch_atlas.py
uv run python scripts/fetch_census.py
```

In [1]:
import io
import json
import zipfile
from datetime import date
from pathlib import Path

import polars as pl

# Find data/raw/ whether the notebook runs from notebooks/ or the project root
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"

# Show every column and cut long text short when printing tables
# Docs: https://docs.pola.rs/api/python/stable/reference/api/polars.Config.set_tbl_cols.html
#       https://docs.pola.rs/api/python/stable/reference/api/polars.Config.set_fmt_str_lengths.html
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(40);  # ; hides the output

## 1. OSM ALPR Cameras (primary)

Source: OpenStreetMap, pulled with the Overpass API by `scripts/fetch_alpr_cameras.py`.
File: `data/raw/Illinois_alpr.geojson`

**One row = one ALPR camera.**

In [2]:
# Load: the file is GeoJSON, one "feature" per camera. Each feature has
# geometry.coordinates ([lon, lat], longitude FIRST) and a dict of properties.
# Flatten each camera into one dict, then pl.DataFrame turns the list of dicts into a table.
# Docs: https://docs.python.org/3/library/json.html#json.load
#       https://docs.pola.rs/api/python/stable/reference/dataframe/index.html
with open(RAW / "Illinois_alpr.geojson", encoding="utf-8") as f:
    geojson = json.load(f)

rows = []
for feature in geojson["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    rows.append({"lon": lon, "lat": lat, **feature["properties"]})

cams = pl.DataFrame(rows)

In [3]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
cams.shape

(8047, 11)

In [4]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
cams.schema

Schema([('lon', Float64),
        ('lat', Float64),
        ('osm_id', Int64),
        ('operator', String),
        ('manufacturer', String),
        ('direction', String),
        ('surveillance_type', String),
        ('man_made', String),
        ('timestamp', String),
        ('version', Int64),
        ('all_tags',
         Struct({'brand': String, 'brand:wikidata': String, 'camera:mount': String, 'camera:type': String, 'direction': String, 'man_made': String, 'operator': String, 'operator:wikidata': String, 'surveillance': String, 'surveillance:type': String, 'surveillance:zone': String, 'manufacturer': String, 'manufacturer:wikidata': String, 'operator:short': String, 'addr:street': String, 'colour': String, 'height': String, 'model': String, 'note': String}))])

In [5]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
cams.head()

lon,lat,osm_id,operator,manufacturer,direction,surveillance_type,man_made,timestamp,version,all_tags
f64,f64,i64,str,str,str,str,str,str,i64,struct[19]
-88.126047,42.168572,235538250,null,"""Flock Safety""","""180""","""ALPR""","""surveillance""","""2025-08-23T12:05:17Z""",3,"{null,null,null,""fixed"",""180"",""surveillance"",null,null,""public"",""ALPR"",""traffic"",""Flock Safety"",""Q108485435"",null,null,null,null,null,null}"
-88.153174,41.974905,237644546,null,"""Flock Safety""","""120""","""ALPR""","""surveillance""","""2026-02-10T17:06:28Z""",4,"{null,null,null,""fixed"",""120"",""surveillance"",null,null,""outdoor"",""ALPR"",null,""Flock Safety"",""Q108485435"",null,null,null,null,null,null}"
-87.641906,41.900434,261326106,null,null,"""175""","""ALPR""","""surveillance""","""2025-09-03T16:59:09Z""",6,"{null,null,null,""fixed"",""175"",""surveillance"",null,null,null,""ALPR"",null,null,null,null,null,null,null,null,null}"
-87.95591,41.669111,261350820,null,null,"""180;90""","""ALPR""","""surveillance""","""2026-05-29T06:03:13Z""",8,"{null,null,null,null,""180;90"",""surveillance"",null,null,""traffic"",""ALPR"",""traffic"",null,null,null,null,null,null,null,null}"
-89.065385,42.338791,643776207,null,"""Flock Safety""","""205""","""ALPR""","""surveillance""","""2025-10-27T13:51:01Z""",3,"{null,null,null,""fixed"",""205"",""surveillance"",null,null,null,""ALPR"",""traffic"",""Flock Safety"",""Q108485435"",null,null,null,null,null,null}"


In [6]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
cams.null_count()

lon,lat,osm_id,operator,manufacturer,direction,surveillance_type,man_made,timestamp,version,all_tags
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,7064,643,525,0,4,0,0,0


In [7]:
# Time range: when cameras were LAST edited in OSM (ISO text, which sorts in date order).
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.min.html
cams.select(
    pl.col("timestamp").min().alias("first_edit"),
    pl.col("timestamp").max().alias("last_edit"),
)

first_edit,last_edit
str,str
"""2023-05-15T04:43:27Z""","""2026-09-28T02:57:19Z"""


In [8]:
# Geography: the box that contains every camera.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.select.html
cams.select(
    pl.col("lat").min().alias("lat_min"),
    pl.col("lat").max().alias("lat_max"),
    pl.col("lon").min().alias("lon_min"),
    pl.col("lon").max().alias("lon_max"),
)

lat_min,lat_max,lon_min,lon_max
f64,f64,f64,f64
37.152909,42.504376,-91.413964,-87.52537


## 2. Eyes On Flock (primary)

Source: eyesonflock.com's data endpoint (`/api/v1/data`), saved by `scripts/fetch_eyesonflock.py`.
File: the newest `data/raw/eyesonflock_*.json`. It holds `summary` (national totals) and `portals` (one entry per agency); the table is `portals`.

**One row = one agency's Flock transparency portal.**

In [9]:
# Load: pick the newest snapshot (file names end in YYYY-MM-DD, so sorted() puts it last),
# then turn the "portals" list into a DataFrame.
# Docs: https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob
#       https://docs.pola.rs/api/python/stable/reference/dataframe/index.html
eof_path = sorted(RAW.glob("eyesonflock_*.json"))[-1]
with open(eof_path, encoding="utf-8") as f:
    portals = pl.DataFrame(json.load(f)["portals"])

eof_path.name

'eyesonflock_2026-09-27.json'

In [10]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
portals.shape

(1528, 20)

In [11]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
portals.schema

Schema([('portal_url', String),
        ('city', String),
        ('county', String),
        ('state', String),
        ('population', Int64),
        ('type', String),
        ('total_cameras', Int64),
        ('total_searches', Int64),
        ('data_retention', Int64),
        ('vehicles_captured', Int64),
        ('hotlist_hits', Int64),
        ('hotlist_hit_rate', Float64),
        ('organization_count', Int64),
        ('organizations_shared_with', List(String)),
        ('receiving_organization_count', Int64),
        ('organizations_received_from', List(String)),
        ('prohibited_uses', String),
        ('public_search_audit', Boolean),
        ('data_last_updated', String),
        ('slug', String)])

In [12]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
portals.head()

portal_url,city,county,state,population,type,total_cameras,total_searches,data_retention,vehicles_captured,hotlist_hits,hotlist_hit_rate,organization_count,organizations_shared_with,receiving_organization_count,organizations_received_from,prohibited_uses,public_search_audit,data_last_updated,slug
str,str,str,str,i64,str,i64,i64,i64,i64,i64,f64,i64,list[str],i64,list[str],str,bool,str,str
"""https://transparency.flocksafety.com/-el…","""El Cajon""",null,"""CA""",104909,"""PD""",103,516,30,844585,11902,1.41,655,"[""Acworth GA PD"", ""Ada County ID SO"", … ""Yuba County Sheriffs Office""]",null,[],"""Immigration enforcement, traffic enforce…",false,"""2026-09-23T12:16:48.093Z""","""-el-cajon-pd-ca"""
"""https://transparency.flocksafety.com/-sp…",null,"""Spokane""","""WA""",505505,"""SD""",null,null,21,null,null,null,null,[],null,[],"""Immigration enforcement, traffic enforce…",false,"""2026-09-23T08:05:44.140Z""","""-spokane-county-wa-so"""
"""https://transparency.flocksafety.com/abi…","""Abington""",null,"""MA""",null,"""PD""",6,4,30,124674,231,0.19,0,[],0,[],"""Immigration enforcement, traffic enforce…",true,"""2026-09-23T09:31:38.884Z""","""abington-ma-pd"""
"""https://transparency.flocksafety.com/acc…",null,"""Accomack""","""VA""",33326,"""SD""",43,82,21,192642,251,0.13,null,[],null,[],"""Immigration enforcement, traffic enforce…",false,"""2026-09-22T20:47:21.704Z""","""accomack-county-va-so"""
"""https://transparency.flocksafety.com/acw…","""Acworth""",null,"""GA""",null,"""PD""",306,237,30,null,63392,null,null,[],null,[],"""Immigration enforcement, traffic enforce…",true,"""2026-09-23T13:48:59.654Z""","""acworth-ga-pd"""


In [13]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
portals.null_count()

portal_url,city,county,state,population,type,total_cameras,total_searches,data_retention,vehicles_captured,hotlist_hits,hotlist_hit_rate,organization_count,organizations_shared_with,receiving_organization_count,organizations_received_from,prohibited_uses,public_search_audit,data_last_updated,slug
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,224,1304,0,804,0,110,149,66,315,251,466,524,0,1090,0,70,0,0,0


In [14]:
# Time range: when Eyes On Flock last updated each portal (ISO text, sorts in date order).
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.min.html
portals.select(
    pl.col("data_last_updated").min().alias("first_updated"),
    pl.col("data_last_updated").max().alias("last_updated"),
)

first_updated,last_updated
str,str
"""2026-02-25T08:19:32.955Z""","""2026-09-23T15:14:22.712Z"""


In [15]:
# Geography: how many states, and how many agencies are city-level vs county-level.
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
portals.select(
    pl.col("state").n_unique().alias("states"),
    pl.col("city").is_not_null().sum().alias("city_agencies"),
    pl.col("county").is_not_null().sum().alias("county_agencies"),
)

states,city_agencies,county_agencies
u32,u32,u32
45,1304,224


## 3. EFF Atlas of Surveillance (secondary)

Source: EFF's Atlas of Surveillance CSV export, saved by `scripts/fetch_atlas.py`.
File: the newest `data/raw/atlas_of_surveillance_*.csv`

**One row = one (agency, technology) pair**, e.g. "Rockford Police Department uses Gunshot Detection".

In [16]:
# Load: pl.read_csv reads a CSV file into a DataFrame.
# Docs: https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
atlas_path = sorted(RAW.glob("atlas_of_surveillance_*.csv"))[-1]
atlas = pl.read_csv(atlas_path)

atlas_path.name

'atlas_of_surveillance_2026-09-28.csv'

In [17]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
atlas.shape

(15135, 28)

In [18]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
atlas.schema

Schema([('AOSNUMBER', String),
        ('NEWAOSNUMBER (ORI9)', String),
        ('City', String),
        ('County', String),
        ('State', String),
        ('Agency', String),
        ('Type of LEA', String),
        ('Summary', String),
        ('Type of Juris', String),
        ('Technology', String),
        ('TECH ABV', String),
        ('Vendor', String),
        ('Link 1', String),
        ('Link 1 Snapshot', String),
        ('Link 1 Source', String),
        ('Link 1 Type', String),
        ('Link 1 Date', String),
        ('Link 2', String),
        ('Link 2 Snapshot', String),
        ('Link 2 Source', String),
        ('Link 2 Type', String),
        ('Link 2 Date', String),
        ('Link 3', String),
        ('Link 3 Snapshot', String),
        ('Link 3 Source', String),
        ('Link 3 Type', String),
        ('Link 3 Date', String),
        ('Other Links', String)])

In [19]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
atlas.head()

AOSNUMBER,NEWAOSNUMBER (ORI9),City,County,State,Agency,Type of LEA,Summary,Type of Juris,Technology,TECH ABV,Vendor,Link 1,Link 1 Snapshot,Link 1 Source,Link 1 Type,Link 1 Date,Link 2,Link 2 Snapshot,Link 2 Source,Link 2 Type,Link 2 Date,Link 3,Link 3 Snapshot,Link 3 Source,Link 3 Type,Link 3 Date,Other Links
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""AOS000001""","""VA0850300BWC""","""Woodstock""","""Shenandoah County""","""VA""","""Woodstock Police Department""","""Police""","""The Woodstock Police Department has used…","""Municipal""","""Body-worn Cameras""","""""","""VML Insurance Programs""","""https://www.vrsa.us/body-camera-footage-…","""""","""Virgina Risk Sharing Association""","""""","""12/27/2012""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",""""""
"""AOS000002""","""IL1010400GDT""","""Rockford""","""Winnebago County""","""IL""","""Rockford Police Department""","""Police""","""The Rockford Police Department spent $31…","""Municipal""","""Gunshot Detection""","""""","""ShotSpotter""","""https://www.rrstar.com/news/20181006/les…","""""","""RR Star""","""""","""10/06/2018""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",""""""
"""AOS000003""","""GAAPD0000GDT""","""Atlanta""","""Fulton County""","""GA""","""Atlanta Police Department""","""Police""","""The Atlanta Police Department began usin…","""Municipal""","""Gunshot Detection""","""""","""ShotSpotter""","""https://www.wsbtv.com/news/local/atlanta…","""""","""WSB TV""","""""","""11/16/2018""","""https://www.atlantapd.org/Home/ShowDocum…","""""","""Atlanta Police Department""","""""","""09/15/2018""","""https://www.cbs46.com/news/metro-atlanta…","""""","""CBS 46""","""""","""12/27/2021""",""""""
"""AOS000004""","""IL0720700GDT""","""Peoria""","""Peoria County""","""IL""","""Peoria Police Department""","""Police""","""The Peoria Police Department began using…","""Municipal""","""Gunshot Detection""","""""","""ShotSpotter""","""http://www.icjia.state.il.us/assets/arti…","""""","""icjia.state.il.us""","""""","""12/13/2019""","""https://www.shotspotter.com/news/peoria-…","""""","""Shotspotter""","""""","""03/01/2015""","""https://web.archive.org/web/201503170342…","""""","""Associated Press""","""""","""03/01/2015""",""""""
"""AOS000005""","""NC0960100GDT""","""Goldsboro""","""Wayne County""","""NC""","""Goldsboro Police Department""","""Police""","""The Goldsboro Police Department uses Sho…","""Municipal""","""Gunshot Detection""","""""","""ShotSpotter""","""https://www.cbs17.com/news/goldsboro-pol…","""""","""CBS 17""","""""","""10/23/2017""","""https://www.facebook.com/goldsborodailyn…","""""","""Goldsboro Police Department Facebook Pag…","""""","""4/27/2020""","""""","""""","""""","""""","""""",""""""


In [20]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
atlas.null_count()

AOSNUMBER,NEWAOSNUMBER (ORI9),City,County,State,Agency,Type of LEA,Summary,Type of Juris,Technology,TECH ABV,Vendor,Link 1,Link 1 Snapshot,Link 1 Source,Link 1 Type,Link 1 Date,Link 2,Link 2 Snapshot,Link 2 Source,Link 2 Type,Link 2 Date,Link 3,Link 3 Snapshot,Link 3 Source,Link 3 Type,Link 3 Date,Other Links
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


`null_count()` shows 0 everywhere because this file stores blanks as empty text `""`, not as null.

In [21]:
# Time range: "Link 1 Date" is the date of the evidence, as text like "10/06/2018" or "10/1/18".
# .str.to_date() turns text into a date: %Y = 4-digit year, %y = 2-digit year.
# pl.when().then().otherwise() picks the format; strict=False gives null for values that aren't dates.
# Some dates are typos (year 0005, 3023), so keep 1990 to today only.
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_date.html
#       https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html
text = pl.col("Link 1 Date")
evidence_date = (
    pl.when(text.str.contains(r"^\d{1,2}/\d{1,2}/\d{2}$"))
    .then(text.str.to_date("%m/%d/%y", strict=False))
    .otherwise(text.str.to_date("%m/%d/%Y", strict=False))
)
atlas.select(evidence_date.alias("d")).filter(pl.col("d").is_between(date(1990, 1, 1), date.today())).select(
    pl.col("d").min().alias("earliest"),
    pl.col("d").max().alias("latest"),
    pl.len().alias("rows_with_a_date"),
)

earliest,latest,rows_with_a_date
date,date,u32
2002-05-14,2026-08-14,10418


In [22]:
# Geography: how many different agencies and state/territory codes.
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
atlas.select(
    pl.col("Agency").n_unique().alias("agencies"),
    pl.col("State").n_unique().alias("states"),
)

agencies,states
u32,u32
7762,58


## 4. US Census by ZIP code (secondary)

Source: US Census Bureau, saved by `scripts/fetch_census.py` into `data/raw/census/`:
- `acsdt5y2024-b19013.dat`: ACS 5-year table B19013, median household income
- `acsdt5y2024-b01003.dat`: ACS 5-year table B01003, total population
- `2024_Gaz_zcta_national.zip`: 2024 Gazetteer, land area

The ACS files cover every kind of place (states, counties, tracts, ...); ZIP code rows are the ones whose `GEO_ID` starts with `860Z200US`.
`_E001` = the estimate, `_M001` = its margin of error.

**One row = one ZCTA** (ZIP Code Tabulation Area, the Census's version of a ZIP code).

In [23]:
# Load the two ACS tables (separated by |) and keep only the ZIP code rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
#       https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html
CENSUS = RAW / "census"
income_all = pl.read_csv(CENSUS / "acsdt5y2024-b19013.dat", separator="|")
population_all = pl.read_csv(CENSUS / "acsdt5y2024-b01003.dat", separator="|")

income = income_all.filter(pl.col("GEO_ID").str.starts_with("860Z200US"))
population = population_all.filter(pl.col("GEO_ID").str.starts_with("860Z200US"))

# Load the Gazetteer: a zip holding one tab-separated file.
# GEOID is read as text so ZIPs like "00601" keep their leading zeros.
# Docs: https://docs.python.org/3/library/zipfile.html#zipfile.ZipFile
with zipfile.ZipFile(CENSUS / "2024_Gaz_zcta_national.zip") as z:
    gazetteer = pl.read_csv(io.BytesIO(z.read(z.namelist()[0])), separator="\t", schema_overrides={"GEOID": pl.String})

# The last column name has trailing spaces ("INTPTLONG      "), so strip every column name.
gazetteer.columns = [col.strip() for col in gazetteer.columns]

# Rows before keeping only ZIP code rows (every kind of place)
{"income_all": income_all.shape, "population_all": population_all.shape}

{'income_all': (543579, 3), 'population_all': (616690, 3)}

**Income (B19013), ZIP rows**

In [24]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
income.shape

(33772, 3)

In [25]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
income.schema

Schema([('GEO_ID', String), ('B19013_E001', Int64), ('B19013_M001', Int64)])

In [26]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
income.head()

GEO_ID,B19013_E001,B19013_M001
str,i64,i64
"""860Z200US00601""",19454,1546
"""860Z200US00602""",21420,1811
"""860Z200US00603""",20933,1650
"""860Z200US00606""",20992,3212
"""860Z200US00610""",24496,1378


In [27]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
income.null_count()

GEO_ID,B19013_E001,B19013_M001
u32,u32,u32
0,0,0


**Population (B01003), ZIP rows**

In [28]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
population.shape

(33772, 3)

In [29]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
population.schema

Schema([('GEO_ID', String), ('B01003_E001', Int64), ('B01003_M001', Int64)])

In [30]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
population.head()

GEO_ID,B01003_E001,B01003_M001
str,i64,i64
"""860Z200US00601""",16669,510
"""860Z200US00602""",37233,270
"""860Z200US00603""",48448,1035
"""860Z200US00606""",5163,293
"""860Z200US00610""",25357,355


In [31]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
population.null_count()

GEO_ID,B01003_E001,B01003_M001
u32,u32,u32
0,0,0


**Gazetteer (land area)**

In [32]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
gazetteer.shape

(33791, 7)

In [33]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
gazetteer.schema

Schema([('GEOID', String),
        ('ALAND', Int64),
        ('AWATER', Int64),
        ('ALAND_SQMI', Float64),
        ('AWATER_SQMI', Float64),
        ('INTPTLAT', Float64),
        ('INTPTLONG', String)])

In [34]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
gazetteer.head()

GEOID,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
str,i64,i64,f64,f64,f64,str
"""00601""",166836392,798613,64.416,0.308,18.180555,"""-66.749961 …"
"""00602""",78546711,4428428,30.327,1.71,18.361945,"""-67.175597 …"
"""00603""",88980555,6253316,34.356,2.414,18.457399,"""-67.124867 …"
"""00606""",114825641,12228,44.334,0.005,18.158327,"""-66.932928 …"
"""00610""",96150194,4289688,37.124,1.656,18.295304,"""-67.12518 …"


In [35]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
gazetteer.null_count()

GEOID,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


Time range: ACS 5-year estimates average **2020–2024**; Gazetteer land areas are for **2024**.

In [36]:
# Geography: how many ZCTAs, and the lowest/highest ZIP (00601 is in Puerto Rico).
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
gazetteer.select(
    pl.col("GEOID").n_unique().alias("zctas"),
    pl.col("GEOID").min().alias("lowest_zip"),
    pl.col("GEOID").max().alias("highest_zip"),
)

zctas,lowest_zip,highest_zip
u32,str,str
33791,"""00601""","""99929"""
